# Retrieval-Augmented Generation — Practical Implementation Notebook

**Stack:** OpenAI `text-embedding-3-small` · ChromaDB · Python 3.10+  
**Sections:**
1. Environment setup and sample corpus
2. Chunking strategies — fixed, recursive, sentence, semantic — with visual comparison
3. Building the vector store with ChromaDB
4. Re-ranking strategies — naive top-K, cross-encoder, MMR, reciprocal rank fusion
5. End-to-end RAG query with each strategy

> Run cells top to bottom. Every section is self-contained after setup.



---
## Section 1 — Environment Setup
---



### 1.1 Install dependencies

Install once; skip if already in your environment.


In [1]:
# Run once, then restart kernel if needed
!pip install --quiet openai chromadb sentence-transformers rank-bm25 nltk tiktoken



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1.2 Imports


In [2]:
import os
import re
import math
import json
import textwrap
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import chromadb
from chromadb.utils import embedding_functions

from openai import OpenAI

import tiktoken
from nltk.tokenize import sent_tokenize
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

print('All imports successful.')


All imports successful.


### 1.3 OpenAI client

Set your API key via environment variable or paste directly (do not commit keys to version control).


In [4]:
# Option A: read from environment (recommended)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_API_KEY')

# Option B: paste directly for local experimentation
# OPENAI_API_KEY = 'sk-...'

assert OPENAI_API_KEY, 'Set OPENAI_API_KEY before continuing.'

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = 'text-embedding-3-small'
CHAT_MODEL  = 'gpt-4o-mini'

print(f'Client ready. Embedding model: {EMBED_MODEL}')


Client ready. Embedding model: text-embedding-3-small


### 1.4 Sample corpus

A set of paragraphs on machine learning topics — long enough to make chunking meaningful.


In [5]:
CORPUS = """
Machine learning is a branch of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed. The learning process
begins with data — observations, examples, or instructions — and the system looks for
patterns in that data to make better decisions in the future. Three broad paradigms
define the field: supervised learning, unsupervised learning, and reinforcement learning.

Supervised learning trains a model on labeled examples, meaning each training sample
carries a known output. Regression and classification are the two primary tasks. Linear
regression fits a straight line to continuous targets; logistic regression estimates
class probabilities. Decision trees partition the feature space recursively. Support
vector machines find a maximum-margin hyperplane. Neural networks compose layers of
parameterized transformations and are trained with gradient descent via backpropagation.

Unsupervised learning finds structure in unlabeled data. K-means clustering assigns
points to the nearest centroid and iterates until convergence. Hierarchical clustering
builds a dendrogram of nested clusters. Principal component analysis reduces
dimensionality by projecting data onto orthogonal directions of maximum variance.
Autoencoders learn compact latent representations through an encoder-decoder architecture.
Generative adversarial networks train a generator against a discriminator to produce
realistic synthetic samples.

Reinforcement learning frames the problem as an agent interacting with an environment.
The agent selects actions to maximize cumulative reward. Q-learning maintains a table
of state-action values and updates them with the Bellman equation. Deep Q-networks
replace the table with a neural network and use experience replay and target networks
for stability. Proximal policy optimization is a policy-gradient method that clips
the objective to prevent destructively large updates.

Natural language processing applies machine learning to human language. Tokenization
splits text into words or subword units. Word embeddings such as Word2Vec and GloVe
represent tokens as dense vectors in a shared semantic space. The transformer
architecture, introduced in the paper Attention Is All You Need, replaces recurrence
with self-attention and has become the foundation of modern NLP. BERT pretrains a
bidirectional transformer with masked language modelling and next-sentence prediction.
GPT models use a causal transformer trained to predict the next token and can generate
fluent, coherent text across diverse domains.

Computer vision uses convolutional neural networks to process image data. A convolution
applies a small learned filter across the spatial dimensions of an image, detecting
local patterns such as edges and textures. Pooling layers subsample feature maps to
reduce computation. Residual networks introduced skip connections that allow gradients
to flow through very deep networks without vanishing. Object detection models such as
YOLO divide the image into a grid and predict bounding boxes and class probabilities
simultaneously. Semantic segmentation assigns a class label to every pixel.

Model evaluation requires held-out test data that the model has never seen. Accuracy
measures the fraction of correct predictions. Precision and recall trade off false
positives against false negatives; the F1 score is their harmonic mean. The area under
the receiver operating characteristic curve summarizes performance across all
classification thresholds. Cross-validation partitions the training set into folds and
reports average performance to reduce variance in the estimate.

Overfitting occurs when a model memorizes training data rather than learning general
patterns. Regularization techniques mitigate this: L2 regularization penalizes large
weights; dropout randomly zeroes activations during training; data augmentation
artificially expands the training set. Early stopping halts training when validation
loss begins to rise. Batch normalization normalizes activations within a mini-batch,
accelerating training and providing a mild regularizing effect.

Retrieval-augmented generation combines a parametric language model with a non-parametric
retrieval component. At inference time the system encodes the user query into a dense
vector, retrieves the most relevant document chunks from a vector store, and concatenates
them with the query before passing the full context to the language model. This approach
grounds the model in external knowledge, reduces hallucination, and makes the knowledge
base updatable without retraining. ChromaDB, FAISS, and Pinecone are popular vector
stores used in production RAG pipelines.
"""

word_count = len(CORPUS.split())
print(f'Corpus loaded: {word_count} words')


Corpus loaded: 645 words



---
## Section 2 — Chunking Strategies
---



### Overview

Chunking determines the retrieval granularity. Chunks that are too large carry noise;
chunks that are too small lose context. This section implements and compares four strategies:

| Strategy | Mechanism | Best for |
|---|---|---|
| Fixed character | Split every N chars with overlap | Quick baseline |
| Recursive character | Prefer natural boundaries | General purpose |
| Sentence-aware | Split on sentence boundaries | Prose, articles |
| Semantic | Split when topic shifts | Complex mixed documents |


### 2.1 Helper — token counter and chunk display


In [6]:
TOKENIZER = tiktoken.encoding_for_model('gpt-4o')

def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))


def display_chunks(chunks: list[str], strategy_name: str, max_preview: int = 80) -> None:
    """Print a structured summary table of chunks."""
    print(f'\n{"-" * 72}')
    print(f'  Strategy : {strategy_name}')
    print(f'  Total    : {len(chunks)} chunks')
    token_counts = [count_tokens(c) for c in chunks]
    print(f'  Tokens   : min={min(token_counts)}  max={max(token_counts)}  avg={sum(token_counts)/len(token_counts):.1f}')
    print(f'{"-" * 72}')
    print(f'  {"#":<4} {"Tokens":<8} {"Preview"}')
    print(f'  {"-"*4} {"-"*8} {"-"*56}')
    for i, (chunk, tc) in enumerate(zip(chunks, token_counts)):
        preview = chunk.replace('\n', ' ').strip()[:max_preview]
        if len(chunk.replace('\n', ' ').strip()) > max_preview:
            preview += '...'
        print(f'  {i:<4} {tc:<8} {preview}')
    print(f'{"-" * 72}\n')


### 2.2 Strategy A — Fixed character chunking

The simplest approach: slice every `chunk_size` characters with a fixed overlap.
No respect for word or sentence boundaries — fast to implement, worst quality.


In [7]:
def chunk_fixed(text: str, chunk_size: int = 400, overlap: int = 60) -> list[str]:
    """
    Split text into fixed-character chunks with overlap.

    Parameters
    ----------
    text       : Input string.
    chunk_size : Characters per chunk.
    overlap    : Characters shared between consecutive chunks.
    """
    chunks = []
    start  = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]


chunks_fixed = chunk_fixed(CORPUS, chunk_size=400, overlap=60)
display_chunks(chunks_fixed, 'Fixed character (size=400, overlap=60)')



------------------------------------------------------------------------
  Strategy : Fixed character (size=400, overlap=60)
  Total    : 14 chunks
  Tokens   : min=65  max=82  avg=72.9
------------------------------------------------------------------------
  #    Tokens   Preview
  ---- -------- --------------------------------------------------------
  0    73       Machine learning is a branch of artificial intelligence that enables systems to ...
  1    71       ms define the field: supervised learning, unsupervised learning, and reinforceme...
  2    71       n estimates class probabilities. Decision trees partition the feature space recu...
  3    69       tering assigns points to the nearest centroid and iterates until convergence. Hi...
  4    73       chitecture. Generative adversarial networks train a generator against a discrimi...
  5    76       with the Bellman equation. Deep Q-networks replace the table with a neural netwo...
  6    82       okenization splits text int

### 2.3 Strategy B — Recursive character chunking

Tries a hierarchy of separators: double newline -> single newline -> space -> character.
Produces cleaner splits that respect paragraph and sentence boundaries.
This is the default strategy in LangChain's `RecursiveCharacterTextSplitter`.


In [8]:
def chunk_recursive(
    text: str,
    chunk_size: int = 400,
    overlap: int   = 60,
    separators: list[str] = None,
) -> list[str]:
    """
    Recursively split text using a priority list of separators.

    The function tries each separator in order. If a resulting piece is still
    larger than chunk_size it recurses with the next separator.
    """
    if separators is None:
        separators = ['\n\n', '\n', '. ', ' ', '']

    def _split(txt: str, seps: list[str]) -> list[str]:
        if not seps or len(txt) <= chunk_size:
            return [txt] if txt.strip() else []
        sep = seps[0]
        parts = txt.split(sep) if sep else list(txt)
        results = []
        current = ''
        for part in parts:
            candidate = (current + sep + part).strip() if current else part.strip()
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    results.append(current)
                if len(part) > chunk_size:
                    results.extend(_split(part, seps[1:]))
                    current = ''
                else:
                    current = part.strip()
        if current:
            results.append(current)
        return results

    raw = _split(text, separators)

    # Apply overlap by prepending tail of previous chunk
    final = []
    for i, chunk in enumerate(raw):
        if i > 0 and overlap > 0:
            prev_tail = final[-1][-overlap:] if final else ''
            chunk = (prev_tail + ' ' + chunk).strip()
        final.append(chunk)
    return [c for c in final if c]


chunks_recursive = chunk_recursive(CORPUS, chunk_size=400, overlap=60)
display_chunks(chunks_recursive, 'Recursive character (size=400, overlap=60)')



------------------------------------------------------------------------
  Strategy : Recursive character (size=400, overlap=60)
  Total    : 18 chunks
  Tokens   : min=28  max=79  avg=57.3
------------------------------------------------------------------------
  #    Tokens   Preview
  ---- -------- --------------------------------------------------------
  0    61       Machine learning is a branch of artificial intelligence that enables systems to ...
  1    28       o make better decisions in the future. Three broad paradigms define the field: s...
  2    69       learning, unsupervised learning, and reinforcement learning. Supervised learning...
  3    38       ision trees partition the feature space recursively. Support vector machines fin...
  4    72       s and are trained with gradient descent via backpropagation. Unsupervised learni...
  5    44       jecting data onto orthogonal directions of maximum variance. Autoencoders learn ...
  6    72       inst a discriminator to

### 2.4 Strategy C — Sentence-aware chunking

Uses NLTK to tokenize sentences, then greedily accumulates them until the
token budget is reached. Guarantees no sentence is split mid-way.


In [9]:
def chunk_by_sentence(
    text: str,
    max_tokens: int = 120,
    overlap_sentences: int = 1,
) -> list[str]:
    """
    Pack complete sentences into chunks without exceeding max_tokens.

    Parameters
    ----------
    text               : Input string.
    max_tokens         : Upper token limit per chunk.
    overlap_sentences  : Number of sentences to repeat at the start of each chunk.
    """
    sentences = sent_tokenize(text.strip())
    chunks    = []
    current   = []
    cur_toks  = 0

    for sentence in sentences:
        s_toks = count_tokens(sentence)
        if cur_toks + s_toks > max_tokens and current:
            chunks.append(' '.join(current))
            # Keep last N sentences as overlap
            current  = current[-overlap_sentences:] if overlap_sentences else []
            cur_toks = sum(count_tokens(s) for s in current)
        current.append(sentence)
        cur_toks += s_toks

    if current:
        chunks.append(' '.join(current))

    return chunks


chunks_sentence = chunk_by_sentence(CORPUS, max_tokens=120, overlap_sentences=1)
display_chunks(chunks_sentence, 'Sentence-aware (max_tokens=120, overlap=1 sentence)')



------------------------------------------------------------------------
  Strategy : Sentence-aware (max_tokens=120, overlap=1 sentence)
  Total    : 9 chunks
  Tokens   : min=101  max=121  avg=110.2
------------------------------------------------------------------------
  #    Tokens   Preview
  ---- -------- --------------------------------------------------------
  0    108      Machine learning is a branch of artificial intelligence that enables systems to ...
  1    108      Regression and classification are the two primary tasks. Linear regression fits ...
  2    107      Hierarchical clustering builds a dendrogram of nested clusters. Principal compon...
  3    110      Q-learning maintains a table of state-action values and updates them with the Be...
  4    111      Word embeddings such as Word2Vec and GloVe represent tokens as dense vectors in ...
  5    121      Computer vision uses convolutional neural networks to process image data. A conv...
  6    101      Model evalua

### 2.5 Strategy D — Semantic chunking

Embeds every sentence, computes cosine similarity between adjacent sentences,
and inserts a boundary wherever similarity drops below a threshold.
Most expensive but produces the most topically coherent chunks.


In [10]:
import numpy as np

def embed_texts(texts: list[str]) -> list[list[float]]:
    """Call OpenAI embedding endpoint in one batch."""
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def chunk_semantic(
    text: str,
    similarity_threshold: float = 0.82,
    max_tokens: int = 200,
) -> list[str]:
    """
    Embed each sentence and create a new chunk whenever adjacent similarity
    falls below similarity_threshold OR the token budget is exceeded.

    Parameters
    ----------
    text                  : Input string.
    similarity_threshold  : Cosine similarity below which a boundary is inserted.
    max_tokens            : Hard token cap per chunk regardless of similarity.
    """
    sentences = sent_tokenize(text.strip())
    if not sentences:
        return []

    print(f'  Embedding {len(sentences)} sentences for semantic chunking...')
    embeddings = embed_texts(sentences)

    chunks   = []
    current  = [sentences[0]]
    cur_toks = count_tokens(sentences[0])

    for i in range(1, len(sentences)):
        sim  = cosine_similarity(embeddings[i - 1], embeddings[i])
        toks = count_tokens(sentences[i])

        if sim < similarity_threshold or cur_toks + toks > max_tokens:
            chunks.append(' '.join(current))
            current  = [sentences[i]]
            cur_toks = toks
        else:
            current.append(sentences[i])
            cur_toks += toks

    if current:
        chunks.append(' '.join(current))

    return chunks


chunks_semantic = chunk_semantic(CORPUS, similarity_threshold=0.82, max_tokens=200)
display_chunks(chunks_semantic, 'Semantic (threshold=0.82, max_tokens=200)')


  Embedding 45 sentences for semantic chunking...

------------------------------------------------------------------------
  Strategy : Semantic (threshold=0.82, max_tokens=200)
  Total    : 45 chunks
  Tokens   : min=8  max=45  avg=19.2
------------------------------------------------------------------------
  #    Tokens   Preview
  ---- -------- --------------------------------------------------------
  0    23       Machine learning is a branch of artificial intelligence that enables systems to ...
  1    34       The learning process begins with data — observations, examples, or instructions ...
  2    21       Three broad paradigms define the field: supervised learning, unsupervised learni...
  3    21       Supervised learning trains a model on labeled examples, meaning each training sa...
  4    9        Regression and classification are the two primary tasks.
  5    19       Linear regression fits a straight line to continuous targets; logistic regressio...
  6    8        De

### 2.6 Side-by-side comparison

Summarize all four strategies in a single comparison table.


In [ ]:
def strategy_summary(name: str, chunks: list[str]) -> dict:
    toks = [count_tokens(c) for c in chunks]
    return {
        'Strategy'  : name,
        'Chunks'    : len(chunks),
        'Min tokens': min(toks),
        'Max tokens': max(toks),
        'Avg tokens': round(sum(toks) / len(toks), 1),
    }

results = [
    strategy_summary('Fixed character',     chunks_fixed),
    strategy_summary('Recursive character', chunks_recursive),
    strategy_summary('Sentence-aware',      chunks_sentence),
    strategy_summary('Semantic',            chunks_semantic),
]

header = f'  {"Strategy":<24} {"Chunks":>7} {"Min tok":>9} {"Max tok":>9} {"Avg tok":>9}'
print(f'\n{"-" * 62}')
print('  CHUNKING STRATEGY COMPARISON')
print(f'{"-" * 62}')
print(header)
print(f'  {"-"*24} {"-"*7} {"-"*9} {"-"*9} {"-"*9}')
for r in results:
    print(f'  {r["Strategy"]:<24} {r["Chunks"]:>7} {r["Min tokens"]:>9} {r["Max tokens"]:>9} {r["Avg tokens"]:>9}')
print(f'{"-" * 62}\n')


### 2.7 Inspect individual chunks

Print the full text of any chunk for inspection.


In [ ]:
def inspect_chunk(chunks: list[str], index: int, label: str = '') -> None:
    chunk = chunks[index]
    print(f'\n--- {label} | Chunk {index} ---')
    print(f'Tokens: {count_tokens(chunk)}')
    print(f'Characters: {len(chunk)}')
    print('Text:')
    print(textwrap.fill(chunk, width=72, initial_indent='  ', subsequent_indent='  '))
    print()

# Show chunk 2 from each strategy
inspect_chunk(chunks_fixed,     2, 'Fixed')
inspect_chunk(chunks_recursive, 2, 'Recursive')
inspect_chunk(chunks_sentence,  2, 'Sentence')
inspect_chunk(chunks_semantic,  2, 'Semantic')



---
## Section 3 — Building the Vector Store with ChromaDB
---



### 3.1 OpenAI embedding function

Wrap the OpenAI embedding endpoint so ChromaDB can call it automatically.


In [ ]:
class OpenAIEmbeddingFunction(embedding_functions.EmbeddingFunction):
    """ChromaDB-compatible wrapper around the OpenAI embedding endpoint."""

    def __init__(self, model: str = EMBED_MODEL):
        self.model = model

    def __call__(self, input: list[str]) -> list[list[float]]:
        response = client.embeddings.create(model=self.model, input=input)
        return [item.embedding for item in response.data]


oai_ef = OpenAIEmbeddingFunction(EMBED_MODEL)
print(f'Embedding function ready: {EMBED_MODEL}')


### 3.2 Ingest chunks into ChromaDB

We ingest the recursive chunks as the primary collection — a good general-purpose baseline.
Each chunk is stored with metadata so we can trace it back to its strategy and index.


In [ ]:
chroma_client = chromadb.Client()

# Drop collection if it exists (safe re-run)
try:
    chroma_client.delete_collection('rag_demo')
except Exception:
    pass

collection = chroma_client.create_collection(
    name='rag_demo',
    embedding_function=oai_ef,
    metadata={'hnsw:space': 'cosine'},
)

# Use recursive chunks as the retrieval corpus
ACTIVE_CHUNKS = chunks_recursive

print(f'Ingesting {len(ACTIVE_CHUNKS)} chunks...')

collection.add(
    documents=ACTIVE_CHUNKS,
    ids=[f'chunk_{i}' for i in range(len(ACTIVE_CHUNKS))],
    metadatas=[{'index': i, 'strategy': 'recursive'} for i in range(len(ACTIVE_CHUNKS))],
)

print(f'Collection ready: {collection.count()} documents stored.')


### 3.3 Raw retrieval baseline

Retrieve the top-K chunks by cosine similarity — no re-ranking applied.


In [ ]:
def retrieve_raw(query: str, n: int = 6) -> list[dict]:
    """
    Query ChromaDB and return ranked results.

    Returns
    -------
    List of dicts: {'rank', 'score', 'id', 'text'}
    """
    results = collection.query(
        query_texts=[query],
        n_results=n,
        include=['documents', 'distances', 'metadatas'],
    )
    docs      = results['documents'][0]
    distances = results['distances'][0]
    ids       = results['ids'][0]

    return [
        {'rank': i + 1, 'score': round(1 - d, 4), 'id': ids[i], 'text': docs[i]}
        for i, d in enumerate(distances)
    ]


def display_results(results: list[dict], label: str, preview: int = 100) -> None:
    print(f'\n{"-" * 72}')
    print(f'  {label}')
    print(f'{"-" * 72}')
    print(f'  {"Rank":<6} {"Score":<8} {"ID":<12} {"Text preview"}')
    print(f'  {"-"*6} {"-"*8} {"-"*12} {"-"*42}')
    for r in results:
        prev = r['text'].replace('\n', ' ')[:preview]
        if len(r['text']) > preview:
            prev += '...'
        print(f'  {r["rank"]:<6} {r["score"]:<8} {r["id"]:<12} {prev}')
    print(f'{"-" * 72}\n')


DEMO_QUERY = 'How does retrieval-augmented generation reduce hallucination?'
raw_results = retrieve_raw(DEMO_QUERY, n=6)
display_results(raw_results, f'Raw cosine retrieval  |  query: "{DEMO_QUERY}"')



---
## Section 4 — Re-Ranking Strategies
---



### Overview

The initial cosine retrieval scores semantic proximity of embeddings but does not
directly measure relevance for a specific query. Re-ranking applies a secondary
scoring step on the candidate pool to surface the most useful chunks.

| Strategy | Model | Characteristic |
|---|---|---|
| Raw top-K | Cosine similarity | Baseline, fast |
| Cross-encoder | `ms-marco-MiniLM-L-6-v2` | Reads query+doc jointly, most accurate |
| MMR | Cosine similarity + diversity penalty | Reduces redundancy |
| Reciprocal Rank Fusion | BM25 + cosine ranks | Combines keyword and semantic |

> Cross-encoder downloads ~80 MB on first run.


### 4.1 Strategy 1 — Raw top-K (baseline)

No re-ranking. Return whatever ChromaDB returns sorted by cosine similarity.
Already shown above; reproduced here for a clean comparison cell.


In [ ]:
def rerank_topk(query: str, n_retrieve: int = 8, n_return: int = 3) -> list[dict]:
    """
    Retrieve top-N chunks by cosine similarity and return the best n_return.
    No secondary scoring.
    """
    candidates = retrieve_raw(query, n=n_retrieve)
    return candidates[:n_return]


topk_results = rerank_topk(DEMO_QUERY, n_retrieve=8, n_return=3)
display_results(topk_results, 'Strategy 1 -- Raw top-K  (n_return=3)')


### 4.2 Strategy 2 — Cross-encoder re-ranking

A bi-encoder (the embedding model) encodes query and document independently.
A cross-encoder reads the concatenation `[query, document]` and produces a
relevance score directly. Much more accurate but cannot be used for first-pass
retrieval because it requires evaluating every (query, document) pair.

**Pattern:** retrieve k=8 candidates with cosine, re-rank all 8, return top 3.


In [ ]:
print('Loading cross-encoder model (first run downloads ~80 MB)...')
CROSS_ENCODER = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Cross-encoder ready.')


In [ ]:
def rerank_cross_encoder(
    query: str,
    n_retrieve: int = 8,
    n_return: int = 3,
) -> list[dict]:
    """
    Retrieve with cosine, re-score each (query, doc) pair with a cross-encoder,
    return the top n_return by cross-encoder score.
    """
    candidates = retrieve_raw(query, n=n_retrieve)
    pairs      = [(query, c['text']) for c in candidates]
    ce_scores  = CROSS_ENCODER.predict(pairs)

    for i, c in enumerate(candidates):
        c['ce_score'] = round(float(ce_scores[i]), 4)
        c['original_rank'] = c['rank']

    reranked = sorted(candidates, key=lambda x: x['ce_score'], reverse=True)
    for i, r in enumerate(reranked):
        r['rank'] = i + 1

    return reranked[:n_return]


ce_results = rerank_cross_encoder(DEMO_QUERY, n_retrieve=8, n_return=3)
display_results(ce_results, 'Strategy 2 -- Cross-encoder  (n_return=3)')

# Show score detail
print('  Cross-encoder score detail:')
for r in ce_results:
    print(f'    rank={r["rank"]}  ce_score={r["ce_score"]:>8.4f}  cosine={r["score"]}  was_rank={r["original_rank"]}')


### 4.3 Strategy 3 — Maximal Marginal Relevance (MMR)

MMR balances relevance and diversity. After selecting the first chunk (highest
cosine score), each subsequent chunk is chosen to maximize the marginal gain:

```
MMR score = lambda * sim(chunk, query) - (1 - lambda) * max_sim(chunk, selected)
```

`lambda = 1.0` is pure relevance; `lambda = 0.0` is pure diversity.
Useful when the top-K chunks are near-duplicates of each other.


In [ ]:
def embed_single(text: str) -> list[float]:
    return client.embeddings.create(model=EMBED_MODEL, input=[text]).data[0].embedding


def rerank_mmr(
    query: str,
    n_retrieve: int = 8,
    n_return: int = 3,
    lam: float = 0.6,
) -> list[dict]:
    """
    Maximal Marginal Relevance re-ranking.

    Parameters
    ----------
    query      : User query string.
    n_retrieve : Number of candidates retrieved from vector store.
    n_return   : Number of final chunks to return.
    lam        : Balance between relevance (1.0) and diversity (0.0).
    """
    candidates = retrieve_raw(query, n=n_retrieve)
    query_vec  = embed_single(query)
    doc_vecs   = [embed_single(c['text']) for c in candidates]

    selected_idx  = []
    remaining_idx = list(range(len(candidates)))

    while len(selected_idx) < n_return and remaining_idx:
        best_idx   = None
        best_score = float('-inf')

        for i in remaining_idx:
            rel = cosine_similarity(doc_vecs[i], query_vec)
            if selected_idx:
                redundancy = max(
                    cosine_similarity(doc_vecs[i], doc_vecs[j])
                    for j in selected_idx
                )
            else:
                redundancy = 0.0

            score = lam * rel - (1 - lam) * redundancy
            if score > best_score:
                best_score = score
                best_idx   = i

        selected_idx.append(best_idx)
        remaining_idx.remove(best_idx)

    results = []
    for rank, i in enumerate(selected_idx, 1):
        r = dict(candidates[i])
        r['rank'] = rank
        results.append(r)

    return results


mmr_results = rerank_mmr(DEMO_QUERY, n_retrieve=8, n_return=3, lam=0.6)
display_results(mmr_results, 'Strategy 3 -- MMR  (lambda=0.6, n_return=3)')


### 4.4 Strategy 4 — Reciprocal Rank Fusion (RRF)

RRF combines two independent retrieval signals without requiring score calibration:

```
RRF score = sum( 1 / (k + rank_i) )   for each retrieval system i
```

Here we fuse **cosine similarity** (semantic) and **BM25** (keyword). Documents that
rank highly in both systems receive the strongest combined score. `k=60` is the
standard constant that reduces the influence of very high ranks.


In [ ]:
def rerank_rrf(
    query: str,
    n_retrieve: int = 8,
    n_return: int = 3,
    k: int = 60,
) -> list[dict]:
    """
    Reciprocal Rank Fusion of BM25 (keyword) and cosine (semantic) signals.

    Parameters
    ----------
    query      : User query string.
    n_retrieve : Candidates from each retrieval system.
    n_return   : Final chunks to return.
    k          : RRF constant (default 60 per the original paper).
    """
    # --- Signal 1: cosine retrieval ---
    cosine_hits = retrieve_raw(query, n=n_retrieve)
    cosine_rank = {h['id']: i + 1 for i, h in enumerate(cosine_hits)}
    id_to_text  = {h['id']: h['text'] for h in cosine_hits}

    # --- Signal 2: BM25 keyword retrieval ---
    corpus_ids   = [f'chunk_{i}' for i in range(len(ACTIVE_CHUNKS))]
    tokenized    = [c.lower().split() for c in ACTIVE_CHUNKS]
    bm25         = BM25Okapi(tokenized)
    bm25_scores  = bm25.get_scores(query.lower().split())

    # Top n_retrieve by BM25
    top_bm25_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:n_retrieve]
    bm25_rank    = {corpus_ids[i]: rank + 1 for rank, i in enumerate(top_bm25_idx)}
    for i in top_bm25_idx:
        cid = corpus_ids[i]
        if cid not in id_to_text:
            id_to_text[cid] = ACTIVE_CHUNKS[i]

    # --- Fuse ---
    all_ids = set(cosine_rank) | set(bm25_rank)
    rrf_scores = {}
    for cid in all_ids:
        score = 0.0
        if cid in cosine_rank:
            score += 1 / (k + cosine_rank[cid])
        if cid in bm25_rank:
            score += 1 / (k + bm25_rank[cid])
        rrf_scores[cid] = score

    top_ids = sorted(rrf_scores, key=lambda cid: rrf_scores[cid], reverse=True)[:n_return]

    return [
        {
            'rank'        : rank + 1,
            'score'       : round(rrf_scores[cid], 6),
            'id'          : cid,
            'text'        : id_to_text[cid],
            'cosine_rank' : cosine_rank.get(cid, 'n/a'),
            'bm25_rank'   : bm25_rank.get(cid, 'n/a'),
        }
        for rank, cid in enumerate(top_ids)
    ]


rrf_results = rerank_rrf(DEMO_QUERY, n_retrieve=8, n_return=3, k=60)
display_results(rrf_results, 'Strategy 4 -- Reciprocal Rank Fusion  (n_return=3)')

print('  RRF score breakdown:')
print(f'  {"Rank":<6} {"RRF score":<12} {"Cosine rank":<14} {"BM25 rank"}')
print(f'  {"-"*6} {"-"*12} {"-"*14} {"-"*10}')
for r in rrf_results:
    print(f'  {r["rank"]:<6} {r["score"]:<12} {str(r["cosine_rank"]):<14} {r["bm25_rank"]}')



---
## Section 5 — Side-by-Side Comparison of All Re-Ranking Strategies
---



Run all four strategies against the same query and display results in a unified table.


In [ ]:
def run_all_strategies(query: str, n_retrieve: int = 8, n_return: int = 3) -> None:
    print(f'\nQuery: "{query}"')
    print(f'Retrieving {n_retrieve} candidates, returning top {n_return} per strategy.\n')

    strategies = [
        ('Raw top-K',        rerank_topk(query, n_retrieve, n_return)),
        ('Cross-encoder',    rerank_cross_encoder(query, n_retrieve, n_return)),
        ('MMR (lam=0.6)',    rerank_mmr(query, n_retrieve, n_return, lam=0.6)),
        ('RRF (BM25+cos)',   rerank_rrf(query, n_retrieve, n_return)),
    ]

    for name, results in strategies:
        print(f'  [{name}]')
        for r in results:
            preview = r['text'].replace('\n', ' ')[:90]
            if len(r['text']) > 90:
                preview += '...'
            print(f'    {r["rank"]}. (score={r["score"]})  {preview}')
        print()


run_all_strategies(DEMO_QUERY)


### Try your own query

Change the string below and re-run to test any question against the corpus.


In [ ]:
MY_QUERY = 'What is the difference between supervised and unsupervised learning?'

run_all_strategies(MY_QUERY, n_retrieve=8, n_return=3)



---
## Section 6 — End-to-End RAG Query
---



Plug any re-ranking strategy into a complete RAG answer generation pipeline.
The retrieved chunks are injected as context before the user question.


In [ ]:
SYSTEM_PROMPT = (
    'You are a precise technical assistant. '
    'Answer the user question using ONLY the provided context. '
    'If the answer is not present in the context, state that clearly. '
    'Do not add information from outside the context.'
)


def rag_answer(
    query: str,
    strategy: str = 'cross_encoder',
    n_retrieve: int = 8,
    n_return: int = 3,
) -> str:
    """
    Full RAG pipeline: retrieve -> re-rank -> generate.

    Parameters
    ----------
    query      : User question.
    strategy   : One of 'topk', 'cross_encoder', 'mmr', 'rrf'.
    n_retrieve : Candidate pool size.
    n_return   : Chunks passed to the LLM.
    """
    strategy_map = {
        'topk'         : lambda q: rerank_topk(q, n_retrieve, n_return),
        'cross_encoder': lambda q: rerank_cross_encoder(q, n_retrieve, n_return),
        'mmr'          : lambda q: rerank_mmr(q, n_retrieve, n_return),
        'rrf'          : lambda q: rerank_rrf(q, n_retrieve, n_return),
    }
    assert strategy in strategy_map, f'Unknown strategy: {strategy}'

    chunks = strategy_map[strategy](query)
    context = '\n\n---\n\n'.join(c['text'] for c in chunks)

    user_message = (
        f'Context:\n{context}\n\n'
        f'Question: {query}'
    )

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_message},
        ],
        temperature=0.1,
        max_tokens=512,
    )
    return response.choices[0].message.content


# Run with cross-encoder strategy
answer = rag_answer(DEMO_QUERY, strategy='cross_encoder')
print(f'Query: {DEMO_QUERY}\n')
print('Answer:')
print(textwrap.fill(answer, width=72))


### Compare answers across strategies


In [ ]:
comparison_query = 'What regularization techniques prevent overfitting?'

for strat in ['topk', 'cross_encoder', 'mmr', 'rrf']:
    print(f'\n--- Strategy: {strat} ---')
    ans = rag_answer(comparison_query, strategy=strat)
    print(textwrap.fill(ans, width=72))
